Setup

In [1]:
import os, shutil, subprocess, sys, torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")

for path in ["/kaggle/working", "/kaggle/tmp", "/tmp"]:
    if os.path.exists(path):
        total, used, free = shutil.disk_usage(path)
        print(f"Disk {path}: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java status:", "OK" if java_check.returncode == 0 else "MISSING")
if java_check.stderr:
    print(" ", java_check.stderr.splitlines()[0])

Python: 3.13.16
PyTorch: 2.11.0+cu130 | CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.64 GB
  GPU 1: Tesla T4 | 15.64 GB
Disk /kaggle/working: 20.94 GB free / 20.96 GB total
Disk /tmp: 2700.00 GB free / 8758.39 GB total
Java status: OK
  openjdk version "21.0.12.1" 2026-08-18


In [2]:
import os

os.makedirs("/kaggle/tmp/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U peft accelerate huggingface_hub wandb pycocoevalcap easyocr rapidocr onnxruntime open-clip-torch

import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        val = secrets.get_secret(name)
        print(f"Secret {name}: FOUND (len={len(val)})")
    except Exception as e:
        print(f"Secret {name}: MISSING ({type(e).__name__})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
transformers: 5.20.2
peft: 0.22.0
accelerate: 1.16.0
Secret HF_TOKEN: MISSING (BackendError)
Secret WANDB_API_KEY: MISSING (BackendError)


BLIP

In [3]:
import time
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# Create a crisp synthetic test image with visible text (no network dependency)
img = Image.new("RGB", (512, 512), color=(240, 235, 225))
draw = ImageDraw.Draw(img)
draw.rectangle([120, 90, 392, 430], fill=(200, 45, 45), outline=(40, 40, 40), width=4)
draw.rectangle([145, 160, 367, 320], fill=(255, 255, 255))
font = ImageFont.load_default()
draw.text((165, 190), "ORGANIC\nTOMATO SOUP\nNET WT 14 OZ", fill=(20, 20, 20), font=font)

model_id = "Salesforce/blip2-opt-2.7b"
t0 = time.time()
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"Allocated after load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

prompt = "Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:"
inputs = processor(images=img, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)

print("input_ids shape:", inputs["input_ids"].shape)
print("decoded input_ids:", repr(processor.tokenizer.decode(inputs["input_ids"][0])))

torch.cuda.reset_peak_memory_stats(0)
t1 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs,
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
    )
elapsed = time.time() - t1

full_text = processor.batch_decode(out, skip_special_tokens=True)[0]
new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
print(f"Generated output shape: {out.shape} | Time: {elapsed:.2f}s | Peak VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")
print("Raw decoded output:", repr(full_text))

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded in 92.4s
Allocated after load: 7.70 GB
input_ids shape: torch.Size([1, 55])
decoded input_ids: '<image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image></s>Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:'
Generated output shape: torch.Size([1, 75]) | Time: 6.36s | Peak VRAM: 7.85 GB
Raw decoded output: 'Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text: This soup is vegan, gluten-free, non-GMO, and made with organic ingredients\n'


loRA

In [4]:
import torch
from peft import LoraConfig, get_peft_model

# Freeze vision + Q-Former explicitly
for p in model.vision_model.parameters():
    p.requires_grad = False
for p in model.qformer.parameters():
    p.requires_grad = False
model.query_tokens.requires_grad = False
for p in model.language_projection.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model.language_model = get_peft_model(model.language_model, lora_cfg)
model.language_model.print_trainable_parameters()

# Verify LoRA weights are fp32 while base weights are fp16
lora_dtypes = {p.dtype for n, p in model.named_parameters() if p.requires_grad}
print("Trainable param dtypes:", lora_dtypes)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()

# Build 1 training example: prompt + target caption + EOS, prompt tokens masked to -100
prompt_text = "Visible text: ORGANIC TOMATO SOUP. Alt-text:"
target_text = " A red can of organic tomato soup with a white label."

prompt_enc = processor(images=img, text=prompt_text, return_tensors="pt")
full_enc = processor(
    images=img,
    text=prompt_text + target_text + processor.tokenizer.eos_token,
    return_tensors="pt",
)

input_ids = full_enc["input_ids"].to("cuda:0")
attention_mask = full_enc["attention_mask"].to("cuda:0")
pixel_values = full_enc["pixel_values"].to("cuda:0", torch.float16)

labels = input_ids.clone()
prompt_len = prompt_enc["input_ids"].shape[1]
labels[:, :prompt_len] = -100  # mask prompt (and any image placeholder tokens)

torch.cuda.reset_peak_memory_stats(0)
scaler = torch.amp.GradScaler("cuda")

with torch.amp.autocast("cuda", dtype=torch.float16):
    outputs = model(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )
    loss = outputs.loss

scaler.scale(loss).backward()
print(f"Dry-run loss: {loss.item():.4f} (finite={torch.isfinite(loss).item()})")
print(f"Peak VRAM during 1-step train: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")

model.zero_grad(set_to_none=True)

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 5,242,880 || all params: 2,656,921,600 || trainable%: 0.1973
Trainable param dtypes: {torch.float32}
Dry-run loss: 2.4457 (finite=True)
Peak VRAM during 1-step train: 7.83 GB


OCR Benchmark

In [5]:
import time
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

# Make a slightly blurred & tilted label image to test OCR realism
ocr_img = Image.new("RGB", (400, 260), (235, 230, 220))
d = ImageDraw.Draw(ocr_img)
d.text((40, 60), "TYLENOL Extra Strength\n500 mg Acetaminophen\n100 Caplets", fill=(15, 15, 15))
ocr_img = ocr_img.rotate(6, expand=True, fillcolor=(235, 230, 220)).filter(ImageFilter.GaussianBlur(0.6))
ocr_np = np.array(ocr_img)

# 1. EasyOCR (CPU + GPU)
import easyocr

t0 = time.time()
reader_cpu = easyocr.Reader(["en"], gpu=False, verbose=False)
init_easy_cpu = time.time() - t0
t1 = time.time()
res_easy_cpu = reader_cpu.readtext(ocr_np)
lat_easy_cpu = time.time() - t1

t0 = time.time()
reader_gpu = easyocr.Reader(["en"], gpu=True, verbose=False)
init_easy_gpu = time.time() - t0
t1 = time.time()
res_easy_gpu = reader_gpu.readtext(ocr_np)
lat_easy_gpu = time.time() - t1

print(f"[EasyOCR CPU] init={init_easy_cpu:.2f}s | infer={lat_easy_cpu:.3f}s")
print("  texts:", [(t, round(c, 2)) for _, t, c in res_easy_cpu])
print(f"[EasyOCR GPU] init={init_easy_gpu:.2f}s | infer={lat_easy_gpu:.3f}s")

# 2. RapidOCR (ONNXRuntime CPU - reflects free HF Space UX)
from rapidocr import RapidOCR

t0 = time.time()
rapid = RapidOCR()
init_rapid = time.time() - t0
t1 = time.time()
res_rapid = rapid(ocr_np)
lat_rapid = time.time() - t1

print(f"[RapidOCR CPU] init={init_rapid:.2f}s | infer={lat_rapid:.3f}s")
if res_rapid and getattr(res_rapid, "txts", None):
    print("  texts:", [(t, round(float(s), 2)) for t, s in zip(res_rapid.txts, res_rapid.scores)])
else:
    print("  raw:", res_rapid)

[EasyOCR CPU] init=2.77s | infer=1.197s
  texts: [('TYLENOL Extra Stretath', np.float64(0.35)), ('500 mot', np.float64(0.23)), ('100 Caplets', np.float64(0.98)), ('~oaminophan', np.float64(0.05))]
[EasyOCR GPU] init=2.36s | infer=1.170s


[INFO] 2026-11-20 17:49:41,492 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 17:49:41,762 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-20 17:49:41,763 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-20 17:49:42,735 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 17:49:42,742 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-20 17:49:42,743 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-20 17:49:42,790 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 17:49:42,868 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

[RapidOCR CPU] init=0.02s | infer=1.594s
  texts: [('TYLENOL Extra Strength', 1.0), ('500 mg Acetaminophen', 0.98), ('100 Caplets', 1.0)]
